In [1]:
import argilla as rg
from datasets import load_dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import classification_report
import torch
import numpy as np

In [2]:
rg.init(
    api_url="http://localhost:6900",
    api_key="owner.apikey",
    workspace="admin",
)

In [7]:
banking_ds = load_dataset("mteb/banking77")
banking_ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9993
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 3076
    })
})

In [8]:
split = banking_ds["train"].train_test_split(test_size=0.5, seed=42)
to_label1 = split["train"]
to_label2 = split["test"]

len(to_label1), len(to_label2)

(4996, 4997)

In [9]:
sentiment_classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    top_k=None,
)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [10]:
def add_predictions(example):
    preds = sentiment_classifier(example["text"])
    example["predictions"] = preds
    return example

to_label1 = to_label1.map(add_predictions)

Map:   0%|          | 0/4996 [00:00<?, ? examples/s]

In [11]:
settings = rg.TextClassificationSettings(
    label_schema=["POSITIVE", "NEGATIVE"]
)
rg.configure_dataset(
    name="labeling_with_pretrained",
    settings=settings
)

In [14]:
records = []

for example in to_label1.shuffle(seed=42):
    preds = example["predictions"]
    preds = preds[0] if preds and isinstance(preds[0], list) else preds

    record = rg.TextClassificationRecord(
        text=example["text"],
        metadata={"category_id": int(example["label"])},
        prediction=[(pred["label"], float(pred["score"])) for pred in preds],
        prediction_agent="distilbert-base-uncased-finetuned-sst-2-english",
        multi_label=False,
    )
    records.append(record)

len(records)

4996

In [15]:
rg.log(
    name="labeling_with_pretrained",
    records=records
)

Output()

4996 records logged to ]8;id=9476864;http://localhost:6900/datasets/admin/labeling_with_pretrained\http://localhost:6900/datasets/admin/labeling_with_pretrained]8;;\

BulkResponse(dataset='labeling_with_pretrained', processed=4996, failed=0)

In [16]:
rb_dataset = rg.load(
    name="labeling_with_pretrained",
    query="status:Validated"
)

len(rb_dataset)

200

In [17]:
rb_dataset.to_pandas().head()

,text,inputs,prediction,prediction_agent,annotation,annotation_agent,vectors,multi_label,explanation,id,metadata,status,event_timestamp,metrics,search_keywords
0,I'm in China and really need a new card.,{'text': 'I'm in China and really need a new c...,"[(NEGATIVE, 0.9964708089828491), (POSITIVE, 0....",distilbert-base-uncased-finetuned-sst-2-english,NEGATIVE,owner,None,False,None,000b8b95-e080-41cc-a356-2768dfc0fc64,{'category_id': 9},Validated,2026-04-17 13:23:24.550118,{'text_length': 40},None
1,I need a quick transfer from China. How long ...,{'text': 'I need a quick transfer from China. ...,"[(NEGATIVE, 0.9992827773094177), (POSITIVE, 0....",distilbert-base-uncased-finetuned-sst-2-english,NEGATIVE,owner,None,False,None,000bc8d5-3427-4055-a34b-64e65f1477b0,{'category_id': 67},Validated,2026-04-17 13:23:24.569379,{'text_length': 62},None
2,How long will it take to verify my account?,{'text': 'How long will it take to verify my a...,"[(NEGATIVE, 0.9992614388465881), (POSITIVE, 0....",distilbert-base-uncased-finetuned-sst-2-english,NEGATIVE,owner,None,False,None,000e644a-408a-45c8-ab00-ee891f065c0d,{'category_id': 68},Validated,2026-04-17 13:23:24.566664,{'text_length': 43},None
3,My card is about to expire. Do I have to go to...,{'text': 'My card is about to expire. Do I hav...,"[(NEGATIVE, 0.9977705478668213), (POSITIVE, 0....",distilbert-base-uncased-finetuned-sst-2-english,NEGATIVE,owner,None,False,None,000f6a73-f69e-4012-b061-6211523d812f,{'category_id': 9},Validated,2026-04-17 13:23:24.457322,{'text_length': 76},None
4,I need to know what is going on. I'm attemptin...,{'text': 'I need to know what is going on. I'm...,"[(NEGATIVE, 0.9996238946914673), (POSITIVE, 0....",distilbert-base-uncased-finetuned-sst-2-english,NEGATIVE,owner,None,False,None,0011ad79-2e76-4334-857f-7169a8f81357,{'category_id': 35},Validated,2026-04-17 13:23:24.402609,{'text_length': 217},None


In [18]:
train_records = [r for r in rb_dataset if r.annotation is not None]
len(train_records)

200

In [19]:
train_texts = [r.text for r in train_records]
train_labels = [r.annotation for r in train_records]

set(train_labels), len(train_texts)

({'NEGATIVE', 'POSITIVE'}, 200)

In [20]:
label2id = {"NEGATIVE": 0, "POSITIVE": 1}
id2label = {0: "NEGATIVE", 1: "POSITIVE"}

y_train = [label2id[label] for label in train_labels]

In [21]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128,
)

In [22]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

In [23]:
train_dataset = TextDataset(encodings, y_train)

In [24]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [44]:
import importlib
import transformers.training_args as ta

ta = importlib.reload(ta)

training_args = ta.TrainingArguments(
    output_dir="./sentiment-banking-model",
    per_device_train_batch_size=8,
    num_train_epochs=2,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

In [47]:
import importlib
import transformers.trainer as trainer_module

trainer_module = importlib.reload(trainer_module)

trainer = trainer_module.Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

In [50]:
from accelerate.utils import extract_model_from_parallel

# Fix kernel stale state: ensure save_pretrained globals can unwrap wrapped models.
if "extract_model_from_parallel" not in model.save_pretrained.__globals__:
    model.save_pretrained.__globals__["extract_model_from_parallel"] = extract_model_from_parallel

trainer.train()

/Users/joalvarez/dev/github/public/joralvarez/uni_pnl/.ml-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
20,0.000381
40,0.000133


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/joalvarez/dev/github/public/joralvarez/uni_pnl/.ml-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=50, training_loss=0.00022858673706650735, metrics={'train_runtime': 5.6027, 'train_samples_per_second': 71.395, 'train_steps_per_second': 8.924, 'total_flos': 6830350243200.0, 'train_loss': 0.00022858673706650735, 'epoch': 2.0})

## Segunda ronda de predicciones

In [53]:
finetuned_classifier = pipeline(
    task="sentiment-analysis",
    model="./sentiment-banking-model/checkpoint-50",
    tokenizer=tokenizer,
    top_k=None,
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [54]:
def add_finetuned_predictions(example):
    preds = finetuned_classifier(example["text"])
    example["predictions"] = preds
    return example

to_label2 = to_label2.map(add_finetuned_predictions)

Map:   0%|          | 0/4997 [00:00<?, ? examples/s]

In [55]:
settings_2 = rg.TextClassificationSettings(
    label_schema=["POSITIVE", "NEGATIVE"]
)
rg.configure_dataset(
    name="labeling_second_round",
    settings=settings_2
)

In [57]:
records_2 = []

for example in to_label2.shuffle(seed=42):
    preds = example["predictions"]
    preds = preds[0] if preds and isinstance(preds[0], list) else preds

    record = rg.TextClassificationRecord(
        text=example["text"],
        metadata={"category_id": int(example["label"])},
        prediction=[(pred["label"], float(pred["score"])) for pred in preds],
        prediction_agent="sentiment-banking-model-v1",
        multi_label=False,
    )
    records_2.append(record)

len(records_2)

4997

In [58]:
rg.log(
    name="labeling_second_round",
    records=records_2
)

Output()

4997 records logged to ]8;id=9476867;http://localhost:6900/datasets/admin/labeling_second_round\http://localhost:6900/datasets/admin/labeling_second_round]8;;\

BulkResponse(dataset='labeling_second_round', processed=4997, failed=0)

In [59]:
rg.load("labeling_with_pretrained")

                                                   text  \
0              I'm in China and really need a new card.   
1     I need a quick transfer from China.  How long ...   
2           How long will it take to verify my account?   
3     My card is about to expire. Do I have to go to...   
4     I need to know what is going on. I'm attemptin...   
...                                                 ...   
4991        Whats the maximum disposable virtual cards?   
4992        I have a charge for something I didn't buy.   
4993               Why isn't my refund on my statement?   
4994         How do I report my card as lost or stolen?   
4995  Can I use my social security card as a form of...   

                                                 inputs  \
0     {'text': 'I'm in China and really need a new c...   
1     {'text': 'I need a quick transfer from China. ...   
2     {'text': 'How long will it take to verify my a...   
3     {'text': 'My card is about to expire. Do I hav...